In [0]:
%sql
-- createing the bronze.stations by CTAS with nmetadata columns
create or replace table vp_by20021205.bronze.stations as
SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingestion_at
FROM read_files('/Volumes/vp_by20021205/landing/raw/stations/stations.csv', format => 'csv')

In [0]:
%sql
-- createing the bronze.tariffs by CTAS with metadata columns
create or replace table vp_by20021205.bronze.tariffs as
SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingestion_at
FROM read_files('/Volumes/vp_by20021205/landing/raw/tariffs/tariffs.csv', format => 'csv')

In [0]:
%sql
-- table creation code

create table if not exists vp_by20021205.bronze.members;

In [0]:
%sql
-- ingestion into the bronze.members by copy into with metadata columns
COPY INTO vp_by20021205.bronze.members
FROM (
  SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingestion_at
  FROM '/Volumes/vp_by20021205/landing/raw/members/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
# Add change_effective_date column only if it doesn't already exist
_table = "vp_by20021205.bronze.members"
_col = "change_effective_date"

if _col not in [f.name for f in spark.table(_table).schema.fields]:
    spark.sql(f"ALTER TABLE {_table} ADD COLUMNS ({_col} STRING)")
    print(f"Column '{_col}' added to {_table}")
else:
    print(f"Column '{_col}' already exists in {_table}, skipping")

In [0]:
%sql
create table if not exists vp_by20021205.bronze.chargers;

In [0]:
from pyspark.sql.functions import col, current_timestamp

# Read JSON array files (multi-line JSON) for chargers
chargers_df = (
    spark.read
    .option("multiLine", "true")
    .json("/Volumes/vp_by20021205/landing/raw/chargers/")
)

# Add ingestion metadata
chargers_df = (
    chargers_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingestion_at", current_timestamp())
)

# Write to bronze table (overwrite to replace previously ingested bad data)
chargers_df.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable("vp_by20021205.bronze.chargers")

In [0]:
# reader api for charger
# df = (
#     spark.read
#     .option("multiLine", "true")
#     .json("/Volumes/vp_by20021205/landing/raw/chargers/chargers_2026-04-01.json")
# )

In [0]:
# from pyspark.sql.functions import col, current_timestamp
# # Add ingestion metadata
# df = (
#     df
#     .withColumn("_source_file", col("_metadata.file_path"))
#     .withColumn("_ingested_at", current_timestamp())
# )

# # writer api for chargers
# df.write.mode("append").saveAsTable("vp_by20021205.bronze.chargers")

In [0]:
from pyspark.sql.functions import col, current_timestamp

# Source JSON files
source_path = "/Volumes/vp_by20021205/landing/raw/sessions/"

# Stream state volume: ops.pipeline_state
schema_path = (
    "/Volumes/vp_by20021205/ops/pipeline_state/sessions_schema"
)

checkpoint_path = (
    "/Volumes/vp_by20021205/ops/pipeline_state/sessions_checkpoint"
)

target_table = "vp_by20021205.bronze.sessions"

#1. Auto Loader: discover and read new JSON files
sessions_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

In [0]:
# 2. Add ingestion metadata
sessions_df = (
    sessions_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

In [0]:
# 3. Write incrementally and stop after available files
query = (
    sessions_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)
query.awaitTermination()

In [0]:
%sql
-- counts
select count(*) count from vp_by20021205.bronze.chargers;

In [0]:
%sql
select count(*) count from vp_by20021205.bronze.members;

In [0]:
%sql
select count(*) count from vp_by20021205.bronze.sessions;

In [0]:
%sql
select count(*) count from vp_by20021205.bronze.stations;

In [0]:
%sql
select count(*) count from vp_by20021205.bronze.tariffs;